# 🖥️ เทรน G1 เดินไปหยิบของ บน NVIDIA DGX Spark (GB10)

notebook นี้รันบน **Jupyter ของ DGX Spark** (ARM + Blackwell GPU) โดยตรง
— ต่างจาก Colab: ไม่ต้อง mount Drive, ใช้ storage ของเครื่อง (3TB), ไม่มี timeout

**เตรียมครั้งเดียว:** notebook จะ clone repo, แก้ pyproject ให้รองรับ ARM (aarch64),
แล้ว uv sync ให้อัตโนมัติ

> DGX Spark เป็น ARM + GPU ใหม่ (GB10 Blackwell) — cu128 รองรับ Blackwell แต่
> mujoco-warp บน ARM ยังไม่เคยทดสอบในโปรเจคนี้ ถ้า cell เทรน error ให้ส่ง log มา

## 1) ตั้งค่า + เช็คเครื่อง

In [ ]:
import os, subprocess, sys
WORKSPACE = '/home/nexpie/workspace/anun'
REPO = os.path.join(WORKSPACE, 'mjlab-custom')
UV = os.path.expanduser('~/.local/bin/uv')
print('arch:', os.uname().machine)  # aarch64
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2) ติดตั้ง uv (ถ้ายังไม่มี)

In [ ]:
import shutil
if not os.path.exists(UV):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
print('uv:', UV, '->', os.path.exists(UV))

## 3) clone repo (ถ้ายังไม่มี)

In [ ]:
os.makedirs(WORKSPACE, exist_ok=True)
if not os.path.isdir(REPO):
    !cd {WORKSPACE} && git clone -q https://github.com/anunpanya9/mjlab-custom.git
else:
    !cd {REPO} && git pull -q
print('repo:', REPO)

## 4) แก้ pyproject ให้รองรับ ARM (aarch64)

mjlab ตั้ง required-environments เป็น linux x86_64 เท่านั้น — เพิ่ม aarch64
ให้ uv ยอมติดตั้งบน DGX Spark (แก้เฉพาะบนเครื่องนี้ ไม่ push)

In [ ]:
pp = os.path.join(REPO, 'pyproject.toml')
txt = open(pp).read()
arm_line = '  "sys_platform == \'linux\' and platform_machine == \'aarch64\'",'
if 'aarch64' not in txt:
    txt = txt.replace(
        '  "sys_platform == \'linux\' and platform_machine == \'x86_64\'",',
        '  "sys_platform == \'linux\' and platform_machine == \'x86_64\'",\n' + arm_line)
    open(pp,'w').write(txt)
    print('เพิ่ม aarch64 แล้ว')
else:
    print('มี aarch64 อยู่แล้ว')

## 5) uv sync (ติดตั้ง mjlab + torch CUDA + mujoco-warp)

ครั้งแรกดาวน์โหลดหลาย GB ใช้เวลาสักครู่

In [ ]:
!cd {REPO} && {UV} sync 2>&1 | tail -15
print('=== เช็ค torch เห็น GPU ไหม ===')
!cd {REPO} && {UV} run --no-sync python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"

## 6) เทรน! (loco-manip — เดินไปหยิบของ)

- GB10 VRAM เยอะ → ใช้ num-envs เยอะได้ (เริ่ม 2048; ถ้า OOM ลดลง)
- ไม่มี timeout → เทรนยาว 20000 รอบได้สบาย
- เซฟลง workspace (3TB) ไม่ต้อง Drive

**ดู metrics:** `position_error` (ระยะ cube→เป้า) ควรลด, `at_goal` ควรเพิ่ม

In [ ]:
LOG_ROOT = os.path.join(WORKSPACE, 'mjlab_logs')
os.makedirs(LOG_ROOT, exist_ok=True)
!cd {REPO} && {UV} run --no-sync python -m mjlab.scripts.train Mjlab-LocoManip-Unitree-G1 \
    --env.scene.num-envs 2048 \
    --agent.max-iterations 20000 \
    --agent.save-interval 200 \
    --agent.logger tensorboard \
    --log-root {LOG_ROOT}

## 7) หาไฟล์โมเดล .pt

In [ ]:
from pathlib import Path
log_dir = Path(LOG_ROOT) / 'g1_locomanip'
runs = sorted(log_dir.glob('*'), key=os.path.getmtime, reverse=True)
assert runs, 'ไม่พบ run — เทรนสำเร็จหรือยัง?'
ckpts = sorted(runs[0].glob('model_*.pt'), key=lambda p:int(''.join(filter(str.isdigit,p.stem))))
checkpoint = str(ckpts[-1])
print('✅ โมเดล:', checkpoint)

## 8) ทดสอบ — เรนเดอร์วิดีโอ policy ที่เทรนได้

เซฟวิดีโอลง workspace เช่นกัน

In [ ]:
os.environ.setdefault('MUJOCO_GL','egl')
VIDEO_PATH = os.path.join(LOG_ROOT, 'g1_locomanip_spark.mp4')
render_script = f'''
import os; os.environ['MUJOCO_GL']='egl'
import torch, imageio
from dataclasses import asdict
import mjlab.tasks
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper
from mjlab.rl.runner import MjlabOnPolicyRunner
TASK='Mjlab-LocoManip-Unitree-G1'
cfg=load_env_cfg(TASK, play=True); cfg.scene.num_envs=1
env=ManagerBasedRlEnv(cfg=cfg, device='cuda', render_mode='rgb_array')
ag=load_rl_cfg(TASK); rc=load_runner_cls(TASK) or MjlabOnPolicyRunner
w=RslRlVecEnvWrapper(env, clip_actions=ag.clip_actions)
r=rc(w, asdict(ag), device='cuda')
r.load(\'{checkpoint}\', load_cfg={{"actor":True}}, strict=True, map_location='cuda')
pol=r.get_inference_policy(device='cuda')
obs=w.get_observations(); frames=[]
for _ in range(300):
    with torch.inference_mode(): a=pol(obs)
    obs,_,_,_=w.step(a); frames.append(env.render())
imageio.mimsave(\'{VIDEO_PATH}\', frames, fps=30)
print('video ->', \'{VIDEO_PATH}\')
'''
open('/tmp/render_lm.py','w').write(render_script)
!cd {REPO} && {UV} run --no-sync python /tmp/render_lm.py

In [ ]:
from IPython.display import Video
Video(VIDEO_PATH, embed=True, width=480)

## สรุป

โมเดล + วิดีโออยู่ใน `{WORKSPACE}/mjlab_logs/` (storage เครื่อง ไม่หาย)

**เอาไปเล่นบน Mac (บังคับเดินด้วย slider):** ดาวน์โหลด .pt มา แล้ว
```bash
CUDA_VISIBLE_DEVICES='' uv run play Mjlab-LocoManip-Unitree-G1 \
    --checkpoint-file model.pt --viewer viser --num-envs 1
```